# Example 1: Querying and Aggregating Molecular Properties

In this notebook, we go over an example to analyse the data within the `molecules` collection in the `GeckoAP` database. We will go over how to connect to the database, how to query and project the data, and how to use a simple pipeline to transform and aggregate data by outsourcing computation to the database engine. We will consider as our subset the molecules with $\alpha$-pinene as one of their precursor molecules.

## Prerequisites

First, make sure you have **dotenv**, **PyMongo**, and **Pandas** libraries installed in your Python environment. Here's a reminder how to install them using pip:
```bash
pip install python-dotenv
pip install pymongo
pip install pandas
```
Then, ensure you have defined the MongoDB connection string within the `.env` file in the root of this project. It should have a line with the following format within it:
```txt
MONGO_URI=mongodb://<username>:<password>@<hostname>
```
## Step 1: Connecting to the database

Given we have the credentials defined, we may load the `.env` file and access the `Mongo_URI` variable there. We can now create the  `MongoClient` object by passing the URI to it. To connect to the  `molecules` collection in the `GeckoAP` database, we simply pass the database name as key to the client and the collection name as a key to the database:

In [84]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()
uri = os.getenv("MONGO_URI")

db_name = "GeckoAP"
collection_name = "molecules"

client = MongoClient(uri)
db = client[db_name]
collection = db[collection_name]

## Step 2: Defining the query and projection

We are interested in looking at molecules that have $\alpha$-pinene (denoted by `A-pinene` in the data) as one of their precursor molecules. We recall that the precursors are given within the `Parent1` and `Parent2` fields. The fields are arrays of strings, where each string is a precursor. Thus, we need the query $(\text{``A-pinene''} \in \text{Parent1}) \cup (\text{``A-pinene''} \in \text{Parent2})$. We recall that checking whether a value is present in an array field has the same syntax as checking a field with singular values is equal to some given value, and that finding the union of fields happens via the `$or` operator on the individual clauses. Thus, we can define the query as: 

In [85]:
query = {
    "$or": [
        {"Parent1": "A-pinene"},
        {"Parent2": "A-pinene"}
    ]
}

Let's then define the projection of data. We are interested only in the fields `Index`, `Dataset`, `mw`, `PsatNan`, and `PsatSim`. Furthermore, we do not want to include the `_id` field. We can, thus, use field inclusions, with the resulting projection defined as:

In [86]:
projection = {
    "_id": 0,
    "Index": 1,
    "Dataset": 1,
    "mw": 1,
    "PsatNan": 1,
    "PsatSim": 1
}

We can access the data by using the `find` method on the collection, passing the query and projection as arguments. We recall that the result cursor is an iterable that can be iterated over once. We can transform the cursor into a list of documents, where each document is simply a Python dictionary. Let's see what the first document returned looks like:

In [87]:
results = collection.find(query, projection)
documents = list(results)
print(documents[0])

{'Index': 268, 'Dataset': 'Terp', 'mw': 228.32, 'PsatNan': 3.55e-06, 'PsatSim': 4.43e-08}


As we can see, only the fields included in the projection are returned. Getting access to the subset of documents might be all we are interested in, so we could stop here. It is often the case, however, that we want to transform or aggregate the data, which can become a very heavy operation on large datasets. Instead of fetching data and performing these operations client-side, we can take advantage of the aggregation pipeline support in MongoDB, which outsources the computations to the database itself.

## Step 3: Defining the field transformation stage

The saturation vapour pressures in fields `PsatNan` and `PsatSim` are, on average, very small and, furthermore, span a large scale of values. Thus, we may want to transform them into log-scale values. This can be achieved by adding a new field to each document that is computed from existing fields with the `$addFields` stage. In our case we can define use the expression operator `$cond` to define an if-else-clause. In the if-clause, we check the field can be log-transformed i.e. is strictly positive. If the clause holds, we can use the `$log10` operator on the field. Otherwise, we set the value to `None`. Thus, we may define the transform stage as follows:

In [88]:
fields_to_log_transform = ["PsatNan", "PsatSim"]

log_transform_stage = {
    "$addFields": {
        f"log({field})": {
            "$cond": {
                "if": { "$gt": [f"${field}", 0] },
                "then": { "$log10": f"${field}" },
                "else": None 
            }
        } for field in fields_to_log_transform
    }
}

## Step 4: Defining the group stage


Next, we are interested in computing some summary statistics across molecules from different datasets for the `mw` field, as well as the `log(PsatNan)` and `log(PsatSim)` fields computed in the log-transform stage. We want to find the minimum, maximum, and average values, as well as the standard deviation of values, for values categorised by the `Dataset` field. To do this, we can use the `$group` stage. Here, we first define the group key `_id` by passing the `Dataset` as an operator to it (essentially adding `$` at the start of the field name). We can then define a series of new fields to generate by calling accumulators on the existing fields. The accumulator in our case will be `$avg`, `$stdDevSamp`, `$min`, and `$max`. We pass the field to accumulate to each of the operators as an operator (again, adding `$` at the start of the field name). As an example, here's what the stage would look like for computing just the average of the `mw` across the datasets:

```python
example_group_stage = { 
    "$group": {
        "_id": "$Dataset", 
        "avgMW": { "$avg": "$mw" },     
    }
}
```
The full group stage, in our case, can be constructed as follows:

In [ ]:
field_to_group_by = "Dataset"
fields_to_summarise = ["mw", "log(PsatNan)", "log(PsatSim)"]
metrics_to_compute = {
    "avg": "$avg",
    "std": "$stdDevSamp",
    "min": "$min",
    "max": "$max"
}

group_logic = {"_id": f"${field_to_group_by}"}

for field in fields_to_summarise:
    for suffix, operator in metrics_to_compute.items():
        group_logic[f"{suffix}_{field}"] = {operator: f"${field}"}

group_stage = {"$group": group_logic}

## Step 5: Running the pipeline

We may now put all the stages together into a pipeline, i.e. a list of stages. The first stage is querying the data, which is achieved by passing the query dictionary to `$match` stage. Next, we project the data by passing the projection dictionary to `$project` stage. Then, we perform the log-transform stage. Finally, we aggregate the data with the group stage to obtain the final summary statistics:

In [90]:
import pandas as pd

pipeline = [
    { "$match":  query},
    { "$project": projection},
    log_transform_stage,
    group_stage
]

pipeline_results = collection.aggregate(pipeline)
df = pd.DataFrame(pipeline_results).rename(columns={"_id": "Dataset"})
df

,Dataset,avg_mw,std_mw,min_mw,max_mw,avg_log(PsatNan),std_log(PsatNan),min_log(PsatNan),max_log(PsatNan),avg_log(PsatSim),std_log(PsatSim),min_log(PsatSim),max_log(PsatSim)
0,Terp,321.752761,75.683994,74.09,633.55,-12.323736,3.918620,-32.903090,-0.328827,-12.099868,3.003264,-24.510042,-0.614394
1,DTA,358.952482,74.880586,104.12,677.27,-14.934378,4.279522,-36.279014,-2.184422,-14.061766,3.305237,-26.146910,-2.399027


## Extra: Formatting results

The following script formats the above results in a more readable form:

In [93]:
datasets = df["Dataset"].unique()

for ds in datasets:
    ds_df = df[df["Dataset"] == ds].drop(columns=["Dataset"])
    
    long_df = ds_df.melt(var_name="metric_field", value_name="value")
    long_df[["metric", "field"]] = long_df["metric_field"].str.split("_", n=1, expand=True)
    final_df = long_df.pivot(index="field", columns="metric", values="value")

    final_df.columns.name = None
    final_df.index.name = None

    final_df = final_df[["min", "max", "avg", "std"]]
    
    print(f" DATASET: {ds} ".center(50, "="))
    print(final_df.round(3), "\n")

================= DATASET: Terp ==================
                 min      max      avg     std
log(PsatNan) -32.903   -0.329  -12.324   3.919
log(PsatSim) -24.510   -0.614  -12.100   3.003
mw            74.090  633.550  321.753  75.684 

================== DATASET: DTA ==================
                  min      max      avg     std
log(PsatNan)  -36.279   -2.184  -14.934   4.280
log(PsatSim)  -26.147   -2.399  -14.062   3.305
mw            104.120  677.270  358.952  74.881 

